In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import multiprocessing as mp

mp.set_start_method("spawn")

In [3]:
import os
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
# os.environ["MKL_THREADING_LAYER"] = "GNU"

sys.path.append("../02-encoding")

In [4]:
from pkgimp import *

from nb2p import fileop, npop, config, stmodel, database, astparse, dgraph

In [ ]:
TRAIN_DATASET_NAME = "distilkaggle"
TEST_DATASET_NAME = "distilkaggle"

In [12]:
TRAIN_SETUP_NAME = "astn4_256"
TEST_SETUP_NAME = (
    "astn4_256_cross" if TRAIN_DATASET_NAME != TEST_DATASET_NAME else TRAIN_SETUP_NAME
)
TRAIN_SETUP_NAME, TEST_SETUP_NAME

('astn4_256', 'astn4_256_cross')

In [13]:
TRAIN_DIRS = config.dirs(dataset_name=TRAIN_DATASET_NAME)
TRAIN_DIRS.makedirs()
TEST_DIRS = config.dirs(dataset_name=TEST_DATASET_NAME)
TEST_DIRS.makedirs()

making dirs: /home/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /home/haotian/scs/distilkaggle
making dirs: /home/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /home/haotian/scs/distilkaggle/dfgtree
making dirs: /home/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /home/haotian/scs/distilkaggle/logs/full
making dirs: /home/haotian/scs/distilkaggle/models/full
making dirs: /home/haotian/scs/distilkaggle/ipynb
making dirs: /home/haotian/scs-final/distilkaggle/processed-unixcoder-ast
making dirs: /home/haotian/scs-final/distilkaggle
making dirs: /home/haotian/scs-final/distilkaggle/processed-unixcoder-full
making dirs: /home/haotian/scs-final/distilkaggle/dfgtree
making dirs: /home/haotian/scs-final/distilkaggle/processed-unixcoder-eda
making dirs: /home/haotian/scs-final/distilkaggle/logs/full
making dirs: /home/haotian/scs-final/distilkaggle/models/full
making dirs: /home/haotian/scs-final/distilkaggle/ipynb


## Load Data & Pre-processing

### Load dataset

In [14]:
DEVICE = torch.device("cuda")

In [15]:
MAX_LENGTH = 256
MAX_LENGTH

256

Get the list of all samples by globbing the folder

In [24]:
train_samples = glob.glob(str(TEST_DIRS.dfgtree / "train-*.lz4"))
# test_samples = glob.glob(str(TEST_DIRS.dfgtree / "test-*.lz4"))
test_samples = sorted(glob.glob("/home/haotian/scs/distilkaggle/dfgtree/test-*.lz4"))

In [17]:
db, client = database.connect(dataset_name=TEST_DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


Build List of X-y dicts

In [19]:
ids = set(map(
    lambda x: str(x["_id"]),
    db.notebooksegments.find(
        # {"prompted": True, "segment_ends.3": {"$exists": True}, "n_ast_children_of_segments": {"$lte": 256}},
        {"selected": True},
        # {"n_ast_children_of_segments": {"$gt": 1, "$lt": MAX_LENGTH}},
        {"_id": 1},
    ),
))
len(ids)

1024

In [25]:
def clean_segment_ends(segment_ends: List[int]):
    result = set()
    for x in segment_ends:
        if x >= MAX_LENGTH:
            return None, f"segment ends exceed max length: {x}"
        if x not in result and x >= 0:
            result.add(x)

    result = sorted(result)

    if len(result) == 0:
        return None, "empty segment ends after cleaning"

    return result, None

### Build Model Input

In [26]:
def replace_elements(data, replacement_function):
    """
    Recursively replaces elements in an arbitrarily nested list based on a replacement function.

    Args:
        data: The input data, which can be a list or any other iterable.
        replacement_function: A function that takes an element and returns its replacement.

    Returns:
        A new list with replaced elements.
    """

    if not isinstance(data, list):
        return replacement_function(data)

    result = []
    for element in data:
        result.append(replace_elements(element, replacement_function))
    return result

replace_elements([[(0, 1)], [(1, 2), [(2, 3)]]], lambda x: (x[0], x[1] * x[1]))

[[(0, 1)], [(1, 4), [(2, 9)]]]

In [27]:
from dfgtree import DFGNode
import queue
from itertools import chain
from collections.abc import Iterable

def flatten(xs):
    for x in xs:
        # print(x)
        if isinstance(x, tuple):
            for node in x[1]:
                yield (x[0], node)
        elif isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            yield from flatten(x)
        else:
            yield x

def bfs(root: DFGNode):
  """Performs Breadth-First Search on a DFGNode tree.

  Args:
    root: The root node of the DFGNode tree.

  Yields:
    A tuple containing the node's representation and its depth from the root.
  """

  q = queue.Queue()
  q.put((root, 0))

  while not q.empty():
    node, depth = q.get()
    yield node.repr, depth

    for child in node.children:
      q.put((child, depth + 1))

def bfs_layered(root: DFGNode):
    """Performs Breadth-First Search on a DFGNode tree, returning only the layer representations.

    Args:
        root: The root node of the DFGNode tree.

    Returns:
        A list of layers, where each layer is a list of node representations at that depth.
    """

    layers = []
    current_layer = [(0, [root])]
    next_layer = []

    while current_layer:
        nodes = list(flatten([current_layer]))
        if len(nodes) == 0:
            break

        layers.append(current_layer)

        for i, (last_layer_i, node) in enumerate(nodes):
            if node.children:
                next_layer.append((i, node.children))

        current_layer = next_layer
        next_layer = []

    return layers


def build_deep_dataset(samples: List[str]):
    for i, sample in tqdm(enumerate(samples), total=len(samples)):
        # read data
        nb_id = sample.split("/")[-1].split(".")[-2].split("-")[-2]
        if nb_id not in ids:
            # print(f"WARN  ignore {sample}. Reason: should not be included")
            continue
        
        sample_dict = fileop.read_lz4(sample)
        sample_dict["segment_ends"], err = clean_segment_ends(
            sample_dict["segment_ends"]
        )
        if err:
            # print(f"WARN  ignore {sample}. Reason: {err}")
            continue

        sample_dict["y"] = npop.indices_to_binary(
            sample_dict["segment_ends"], sample_dict["segment_ends"][-1] + 1
        )
        
        # extract encodings
        ast_children = []
        for s in sample_dict['segments']:
            node = s['repr']
            ast_children.extend(node.children)

        bfs_result = list(bfs_layered(DFGNode(None, ast_children)))
        bfs_result = bfs_result[1:]
        # pprint(bfs_result)

        if len(bfs_result) == 0:
            # print(f"WARN  ignore {sample}. Reason: encodings is empty")
            continue

        result = list(replace_elements(bfs_result, lambda x: [(x[0], torch.Tensor(b.repr).to(DEVICE)) for b in x[1]]))
        # print(result)
        result = [list(itertools.chain.from_iterable(r)) for r in result]

        code = "\n".join([s['code'] for s in sample_dict['segments']])
        
        yield {
            "code": code,
            "gt_ast": sample_dict["segment_ends"],
            "x": result,
            "y": sample_dict["y"]
        }

# TEST_RESULT = list(build_deep_dataset([train_samples[208], train_samples[209]]))
# print(TEST_RESULT)

In [28]:
test_encodings = list(build_deep_dataset(test_samples))
len(test_encodings), test_encodings[0]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 58513/58513 [00:08<00:00, 6644.52it/s]


(1020,
 {'code': '%matplotlib inline\ncon = sqlite3.connect(\'../input/database.sqlite\')\nprint(pd.read_sql_query("""\nSELECT c.CompetitionName,\n       COUNT(t.Id) NumberOfTeams\nFROM Competitions c\nINNER JOIN Teams t ON t.CompetitionId=c.Id\n-- ONLY including teams that ranked\nWHERE t.Ranking IS NOT NULL\nGROUP BY c.CompetitionName\nORDER BY COUNT(t.Id) DESC\nLIMIT 10\n""", con))\ntop10 = pd.read_sql_query("""\nSELECT *\nFROM Users\nWHERE Ranking IS NOT NULL\nORDER BY Ranking\nLIMIT 10\n""", con)\nprint(top10)\nprint(pd.read_sql_query("""\nSELECT *\nFROM Users\nWHERE HighestRanking=1\n""", con))\nmatplotlib.style.use(\'ggplot\')\ntop10.sort(columns="Points").plot(x="DisplayName", y="Points", kind="barh", color="#20beff")',
  'gt_ast': [1, 2, 3, 5, 6, 8],
  'x': [[(0,
     tensor([[ 9.4532e-01,  7.5682e-01,  1.0616e+00, -6.5870e-02,  3.1204e-02,
              -1.6171e-01,  5.0097e-01,  9.9698e-01, -1.5415e-01,  2.0015e-01,
              -2.4906e-01, -1.1534e+00, -1.4463e+00,  6.414

### Pad to max length

In [29]:
y_test_padded = np.zeros((len(test_encodings), MAX_LENGTH), dtype=np.float32)
y_test_padded.shape

(1020, 256)

In [30]:
### NOTE: An interesting finding that applying multithreading is as expected on parallelism
### but not multiprocessing, as NumPy can only use single CPU core across processes.
MAX_WORKERS = 32

from functools import partial


def assign_data4pool(dest, data):
    i, arr = data
    if len(dest.shape) == 2:
        dest[i, : arr.shape[0]] = arr
    elif len(dest.shape) == 3:
        dest[i, : arr.shape[0], :] = arr


def mask_data4pool(data):
    return npop.mask(len(data['y']), MAX_LENGTH)

In [31]:
y_test_padded_arr = thread_map(
    partial(assign_data4pool, y_test_padded),
    enumerate(d["y"] for d in test_encodings),
    max_workers=MAX_WORKERS,
)

0it [00:00, ?it/s]

In [32]:
test_mask = thread_map(mask_data4pool, test_encodings, max_workers=MAX_WORKERS)
print(test_mask[0])

  0%|          | 0/1020 [00:00<?, ?it/s]

[False False False False False False False False False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  T

In [33]:
test_lengths = list(map(lambda x: (~x).sum(), test_mask))
print(len(test_lengths), test_lengths[0])

1020 9


In [34]:
import gc
import ctypes

del gc.garbage[:]
gc.collect()
libc = ctypes.CDLL("libc.so.6")
libc.malloc_trim(0)

1

## NB2P

In [35]:
torch.autograd.set_detect_anomaly(True)

In [36]:
TEST_X = [e['x'] for e in test_encodings]
CODE_X = [e['code'] for e in test_encodings]

In [37]:
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(42)

### Test

In [38]:
%xdel model
torch.cuda.empty_cache()
gc.collect()
import ctypes

libc = ctypes.CDLL("libc.so.6")
libc.malloc_trim(0)

NameError: name 'model' is not defined


1

In [39]:
import torch
import numpy as np
np.set_printoptions(precision=6, suppress=True)
torch.set_printoptions(precision=6, sci_mode=False)

input_size = MAX_LENGTH
eval_freq = 1
checkpoint_freq = 1

print(
    input_size,
    eval_freq,
    checkpoint_freq,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

256 1 1
cuda


In [62]:
from nb2p.stmodel.dnn import NB2PDecoder, Trainer, DNNInput

MODEL_NAME = "nb2pdecoder"

hidden_size = 512
num_layers = 3
num_internal_layers = 4
epoch = 50

model = NB2PDecoder(
    input_size, 
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_internal_layers=num_internal_layers,
    epsilon_scale=1.0,
    device=device,
    setup='bce',
).to(device)

MODEL_PATH = (
    TEST_DIRS.log
    / f"{TRAIN_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}_il{num_internal_layers}-e{epoch}.pt"
)
print(MODEL_PATH)

model.load_state_dict(torch.load(MODEL_PATH))
model.eval()
print("Model loaded")

/home/haotian/scs-final/distilkaggle/logs/full/astn4_256-nb2pdecoder_bf_ff512_l3_il4-e50.pt
Model loaded


/tmp/ipykernel_1668104/1887124547.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH))


In [63]:
segment_result = Trainer(
    model,
    num_epochs=300,
    batch_size=32,
    eval_freq=eval_freq,
    checkpoint_freq=checkpoint_freq,
    learning_rate=1e-3,
    device=device,
    model_write_dir=TEST_DIRS.log,
    report_interval=50,
    window_size=2000,
    model_name=(
        f"{TRAIN_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}_il{num_internal_layers}"
    ),
    setup='dfgtree-bce',
    # start=50,
).evaluate(
    DNNInput(X=TEST_X, y=y_test_padded, code=CODE_X, mask=test_mask),
    # n_try=10,
)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 32/32 [00:02<00:00, 13.87it/s]


(1020, 256) (1020, 256)
[0, 1, 1, 1, 0, 1, 1, 0]
[0, 1, 0, 1, 0, 1, 1, 1]
[Epoch 0] jaccard: 0.4475944368892706
[Epoch 0] hamming: 14.273529411764706
[Epoch 0] wasserstein: 6.800980392156863
[Epoch 0] precision (micro, avg across samples): 0.6219786030823894
[Epoch 0] recall (micro, avg across samples): 0.6106953604040445
[Epoch 0] f1-score (micro, avg across samples): 0.5794776722386166
[Epoch 0] precision (macro, avg across samples): 0.13709434681873847
[Epoch 0] recall (macro, avg across samples): 0.13709434681873847
[Epoch 0] f1-score (macro, avg across samples): 0.13709434681873847
[Epoch 0] precision (weighted, avg across samples): 0.6106953604040445
[Epoch 0] recall (weighted, avg across samples): 0.6106953604040445
[Epoch 0] f1-score (weighted, avg across samples): 0.6106953604040445
[0, 1, 1, 1, 0, 1, 1, 0]
[0, 1, 0, 1, 0, 1, 1, 1]
Eval Loss = 0.3532, Eval Accuracy = 6.8010


In [64]:
# len(segment_result), segment_result[0]

In [65]:
# fileop.write_json(segment_result, TEST_DIRS.base / f"{TEST_SETUP_NAME}_nb2pss-segment-result.json")

In [66]:
# segment_result = fileop.read_json(TEST_DIRS.base / f"{TEST_SETUP_NAME}_nb2pss-segment-result.json")

In [67]:
segment_result_dict = {i: s for i, s in enumerate(segment_result)}
# print(next(iter(segment_result_dict.values())))

In [68]:
from tqdm.contrib.concurrent import process_map
from functools import partial

MAX_WORKERS = 32

ged_result = process_map(
    partial(dgraph.do_compute_ged, multigraph=False),
    segment_result_dict.items(),
    max_workers=MAX_WORKERS,
    chunksize=1,
)
ged_result_valid = [g for g in ged_result if g is not None]
print("len(ged_result_valid):", len(ged_result_valid))
print("GED:", sum(ged_result_valid) / len(ged_result_valid))

# ged_result = process_map(
#     partial(dgraph.do_compute_ged, multigraph=True),
#     segment_result_dict.items(),
#     max_workers=MAX_WORKERS,
#     chunksize=1,
# )
# ged_result_valid = [g for g in ged_result if g is not None]
# print("len(ged_result_valid):", len(ged_result_valid))
# print("GED (multigraph):", sum(ged_result_valid) / len(ged_result_valid))

  0%|          | 0/1020 [00:00<?, ?it/s]

   8 [7, 12, 17, 28, 36, 45, 51, 57, 70, 85, 86, 89, 91, 96, 109, 114, 127, 140] [7, 12, 17, 28, 36, 45, 51, 57, 70, 89, 91, 94, 96, 99, 109, 112, 114, 118, 122, 127, 140]
  16 [7, 8, 22, 45, 53, 70, 92, 109, 126, 148, 154, 179, 188] [7, 11, 53, 69, 86, 91, 107, 112, 134, 147, 153, 179, 188]
  56 [6, 16, 17, 20, 65, 69, 82, 89, 184] [2, 4, 6, 18, 21, 26, 33, 42, 49, 59, 69, 74, 76, 89, 116, 136, 149, 159, 163, 173, 184]
  14 [4, 7, 10, 17, 21, 23, 25, 27] [4, 7, 10, 13, 17, 27]
  29 [0, 2, 6, 8, 9, 13, 14, 38, 41, 43, 46, 51, 52, 53, 57, 60, 63, 67, 70, 72, 74, 79, 83] [0, 2, 6, 8, 9, 13, 14, 30, 38, 39, 41, 43, 45, 46, 51, 52, 53, 57, 60, 63, 67, 70, 72, 74, 79, 83]
  40 [3, 9, 11, 12, 16, 20, 25, 37, 41, 42, 49, 85, 102, 117, 140] [3, 12, 16, 25, 37, 49, 85, 102, 117, 140]
  65 [5, 6, 11, 16, 18, 21, 24, 27, 32, 33, 55, 58, 98, 116, 146, 151] [5, 11, 16, 18, 23, 24, 27, 32, 40, 49, 55, 56, 58, 77, 82, 92, 96, 98, 116, 122, 128, 134, 140, 146, 151]
  22 [7, 85, 92, 99, 104, 115, 116, 

In [69]:
fileop.write_json(ged_result, TEST_DIRS.base / f"{TEST_SETUP_NAME}_nb2pss-ged.json")